In [89]:
import pandas as pd
import numpy as np
import os

In [90]:
future_before_df = pd.read_csv("../data/before_future_train_v3.csv", index_col=0)
future_train_df = pd.read_csv("../data/future_train_v3.csv", index_col=0)
future_test_df = pd.read_csv("../data/future_test_v3.csv", index_col=0)
future_tot_train_df = pd.concat([future_before_df, future_train_df], axis=0)
future_tot_train_df.reset_index(drop=True, inplace=True)
future_tot_train_df = pd.concat([future_tot_train_df, future_test_df], axis=0)
future_tot_train_df.reset_index(drop=True, inplace=True)
future_tot_train_df.sort_values(by=['date', 'tic_code'], inplace=True)
future_tot_train_df.reset_index(drop=True, inplace=True)

In [91]:
before_df = pd.read_csv("../data/before_train_v3.csv", index_col=0)
train_df = pd.read_csv("../data/train_v3.csv", index_col=0)
test_df = pd.read_csv("../data/test_v3.csv", index_col=0)
tot_train_df = pd.concat([before_df, train_df], axis=0)
tot_train_df.reset_index(drop=True, inplace=True)
tot_train_df = pd.concat([tot_train_df, test_df], axis=0)
tot_train_df.reset_index(drop=True, inplace=True)
tot_train_df.sort_values(by=['date', 'tic_code'], inplace=True)
tot_train_df.reset_index(drop=True, inplace=True)

In [92]:
selected_feat = ['date', 'tic_code', 'close', 'ticker',
       'ret', 'high_ratio', 'low_ratio', 'adjust_close', 'adjust_high',
       'adjust_low', 'return_1m', 'return_3m', 'return_6m', 'return_12m',
       'return_avg', 'mom_12m', 'mom_score', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'SMA_30', 'SMA_60', 'SMA_220',
       'SMA_252', 'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci', 'paa_score',
       'gtaa_buy_signal', 'paa_buy_signal', 'daa_buy_signal',
       'momentum_product', 'dm_buy_signal', 'adjust_close_norm',
       'return_1m_norm', 'return_3m_norm', 'return_6m_norm', 'return_12m_norm',
       'return_avg_norm', 'mom_12m_norm', 'mom_score_norm', 'SMA_30_norm',
       'SMA_60_norm', 'SMA_220_norm', 'SMA_252_norm', 'macd_norm',
       'boll_ub_norm', 'boll_lb_norm', 'rsi_norm', 'cci_norm']

In [93]:
eqaul_df = pd.concat([tot_train_df[selected_feat], future_tot_train_df[selected_feat]])

In [94]:
eqaul_df.sort_values(by=['date', 'tic_code'], inplace=True)
eqaul_df.reset_index(drop=True, inplace=True)

In [95]:
# TRAIN_START_DATE = '2002-02-05'
# TRAIN_END_DATE = '2012-12-31'
# VAILD_START_DATE = '2013-01-01'
# VAILD_END_DATE = '2018-12-31'
# TEST_START_DATE = '2019-01-01'
# TEST_END_DATE = '2024-12-31'

In [96]:
TRAIN_START_DATE = '2002-02-05'
TRAIN_END_DATE = '2012-12-31'

TRADE_START_DATE = '2013-01-01'
TRADE_END_DATE = '2024-12-31'

# TRAIN_START_DATE = '2000-01-10'
# TRAIN_END_DATE = '2023-12-31'
# TRADE_START_DATE = '2010-01-01'
# TRADE_END_DATE = '2023-12-31'

window_days = 252
start_day = 252
top_k = 5
top_pct = 0.5
risk_free_rate = 0
gamma = 10
rebalance_every = 20
cost = 0.003
annual_factor = 252

In [97]:
eqaul_df["ticker"].nunique()

30

In [98]:
equal_price = eqaul_df[["date", "ticker", "close"]].dropna()

In [99]:
equal_df = equal_price.pivot(index="date", columns="ticker", values="close")

In [100]:
equal_df = equal_df.sort_index()

In [101]:
equal_return = equal_df.pct_change().fillna(0)

In [102]:
# ✅ 평균-분산 (Mean-Variance), 최소분산 (MinVar), 리스크패리티 (Risk-Parity), Max Sharpe
# 전략들의 차이는 "최적화 기준"의 차이뿐이며, 백테스트 로직 구조는 거의 동일함

import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

# ✅ 평균-분산 최적화 함수 (공매도 금지)

# ✅ 최소분산 포트폴리오 (Min-Var)
def compute_minvar_weights(mu, cov_matrix):
    n = cov_matrix.shape[0]
    w = cp.Variable(n)
    objective = cp.Minimize(cp.quad_form(w, cov_matrix))
    constraints = [cp.sum(w) == 1, w >= 0]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return w.value


# ✅ 리스크 패리티 가중치 (역분산 기반 근사)
def compute_risk_parity_weight_from_window(returns_window):
    var = returns_window.var()
    inv_var = 1 / var
    weights = inv_var / inv_var.sum()
    return weights


def compute_max_sharpe_min_var(mu, cov_matrix, risk_free_rate=0.0, max_vol=None, target_return=0):
    n = len(mu)
    x = cp.Variable(n)
    excess_mu = mu - risk_free_rate
    # objective = cp.Minimize(cp.quad_form(x, cov_matrix))
    w_tilde = cp.Variable(n)  # 치환된 weight (w_tilde = k * w)
    k = cp.Variable(nonneg=True)  # 스케일 변수
    objective = cp.Minimize(cp.quad_form(w_tilde, cov_matrix))

    # constraints = [cp.sum(x) == 1, x >= 0, excess_mu @ x >= target_return]
    # constraints = [cp.sum(x) == 1, x >= 0, excess_mu @ x == 1]
    constraints = [
        excess_mu.T @ w_tilde == 1,   # 초과수익률 고정
        cp.sum(w_tilde) == k,          # w_tilde = k * w
        w_tilde >= 0          # w_tilde = k * w
    ]


    prob = cp.Problem(objective, constraints)
    prob.solve()
    
    # try:
    #     prob.solve()
    #     if x.value is not None:
    #         return x.value
    #     else:
    #         raise ValueError("No solution from solver")
    # except Exception as e:
    #     return np.ones(n) / n  # fallback: equal weights
    if w_tilde.value is not None and k.value is not None and k.value > 0:
        w = w_tilde.value / k.value
        return w
    else:
        print("최적화 실패. 균등 포트폴리오로 fallback.")
        return np.zeros(n)

# ✅ 백테스트 공통 함수 (최적화 함수 인자로 받음)
def backtest_strategy(returns, compute_weights_fn, rebalance_every=20, window_days=252, cost=0.003, start_day=252, **kwargs):
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = None

    i = start_day
    while i < len(dates) - rebalance_every:
        window_data = returns.iloc[i - window_days:i]
        # print(f"Window data shape: {window_days}")
        if window_data.isna().sum().sum() > 0:
            i += rebalance_every
            continue

        mu = window_data.mean().values
        cov = window_data.cov().values

        # mu = window_data.mean().values
        # cov = window_data.cov().values
        try:
            weights = compute_weights_fn(mu, cov, **kwargs)
        except TypeError:
            weights = compute_weights_fn(window_data, **kwargs)

        if weights is None:
            i += rebalance_every
            continue

        n_assets = returns.shape[1]
        
        if prev_weights is None:
            prev_weights = np.zeros(n_assets)

        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
            
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            if j == 0 and prev_weights is not None:
                turnover = np.abs(weights - prev_weights).sum()
                tc = turnover * cost
            else:
                tc = 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights))

        prev_weights = weights
        i += rebalance_every

    strat_returns = pd.Series(dict(portfolio_returns)).sort_index()
    strat_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return strat_returns, strat_weights


## 다른 전략

In [103]:
# 3. 리스크 패리티 (Risk-Parity)
rp_returns, rp_weights = backtest_strategy(
    equal_return,
    compute_weights_fn=compute_risk_parity_weight_from_window,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost = cost
)

In [104]:
minvar_returns, minvar_weights = backtest_strategy(
    equal_return,
    compute_weights_fn=compute_minvar_weights,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost=cost
)

In [105]:
ms_returns, ms_weights = backtest_strategy(
    equal_return,
    compute_weights_fn=compute_max_sharpe_min_var,
    risk_free_rate=risk_free_rate,  # 무위험 수익률
    target_return=0.0,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost =cost


    # max_vol=0.01   # 최대 변동성 제약 (선형 근사)
)

/opt/conda/lib/python3.11/site-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


최적화 실패. 균등 포트폴리오로 fallback.


In [106]:
def backtest_paa_from_pivot(returns: pd.DataFrame,
                             pivot_score: pd.DataFrame,
                             window_days: int = 252,
                             rebalance_every: int = 20,
                             top_n: int = 10,
                             score_threshold: float = 0.0,
                             cost: float = 0.003,
                             start_day: int = 252):
    """
    PAA 전략 백테스트 (수익률 기반 점수 사용, 거래비용 반영)

    Parameters:
    - returns: 일간 수익률 DataFrame (index: date, columns: tickers)
    - pivot_score: PAA score DataFrame (index: date, columns: tickers)
    - window_days: 과거 데이터 시작 시점
    - rebalance_every: 리밸런싱 주기
    - top_n: 상위 자산 수
    - score_threshold: 점수 필터링 기준
    - cost: 거래 비용 비율

    Returns:
    - paa_returns: 전략 일간 수익률 Series
    - paa_weights: 전략 리밸런싱 시점별 자산 비중 DataFrame
    """
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = pd.Series(0, index=returns.columns, dtype=float)

    i = start_day
    while i < len(dates) - rebalance_every:
        current_date = dates[i]
        score_today = pivot_score.loc[current_date]
        satisfied_assets = score_today[score_today > score_threshold]
        if satisfied_assets.empty:
            selected = []
        else:
            selected = satisfied_assets.sort_values(ascending=False).head(top_n).index

        weights = pd.Series(0, index=returns.columns, dtype=float)
        if len(selected) > 0:
            weights[selected] = 1 / len(selected)

        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
                    
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            tc = np.abs(weights - prev_weights).sum() * cost if j == 0 else 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights.copy()))

        prev_weights = weights.copy()
        i += rebalance_every

    paa_returns = pd.Series(dict(portfolio_returns)).sort_index()
    paa_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return paa_returns, paa_weights

In [107]:
all_paa_score = eqaul_df.pivot(index='date', columns='ticker', values='paa_score')

In [108]:
paa_returns, paa_weights = backtest_paa_from_pivot(
    equal_return, all_paa_score,
    top_n=top_k,
    window_days = window_days,
    rebalance_every=rebalance_every,
    score_threshold=0.0,
    cost=cost
)

In [109]:
def backtest_equal_weight_20day(returns: pd.DataFrame, rebalance_every=20, window_days=252, cost=0.003, start_day=252):
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = None

    i = start_day
    while i < len(dates) - rebalance_every:
        rebalance_day = dates[i]
        window_data = returns.iloc[i - window_days:i]


        n_assets = window_data.shape[1]
        weights = np.ones(n_assets) / n_assets  # 1/N 포트폴리오
        n_assets = returns.shape[1]
        if prev_weights is None:
            prev_weights = np.zeros(n_assets)  # 최초 리밸런싱: 전량 매수로 간주
            
        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]

        for j, (date, row) in enumerate(sub_returns.iterrows()):
            if j == 0 and prev_weights is not None:
                turnover = np.abs(weights - prev_weights).sum()
                tc = turnover * cost
            else:
                tc = 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights))

        prev_weights = weights
        i += rebalance_every

    ew_returns = pd.Series(dict(portfolio_returns)).sort_index()
    ew_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return ew_returns, ew_weights


In [110]:
# 백테스트 실행
equal_returns, equal_weights = backtest_equal_weight_20day(equal_return, rebalance_every=rebalance_every, window_days=window_days, cost=cost)

# Sharpe 비율 계산
# equal_sharpe = equal_returns.mean() / equal_returns.std() * np.sqrt(252)

In [111]:
strategy_returns = {
    'risk_parity': rp_returns,
    'minvar': minvar_returns,
    'max_shapre': ms_returns,
    'paa' : paa_returns,
    'Equal Weight': equal_returns
}

In [112]:
risk_cum = (1 + rp_returns).cumprod()
minvar_cum = (1 + minvar_returns).cumprod()
ms_cum = (1 + ms_returns).cumprod()
paa_cum = (1 + paa_returns).cumprod()
equal_cum = (1 + equal_returns).cumprod()

In [113]:
import os
import sys
# 현재 경로의 부모 디렉토리
parent_dir = os.path.dirname(os.getcwd())

# sys.path에 현재 경로와 부모 경로 추가
sys.path.append(parent_dir)
import eval_metric as em

In [114]:
def calculate_rolling_metrics(
    returns: pd.Series,
    windows: list = [20, 252],
    risk_free_rate: float = 0.02,
    annual_factor: int = 252,
    name : str = "returns",
    shift_features: bool = True,  # ✅ 추가

) -> pd.DataFrame:
    """
    주어진 daily return 시리즈에 대해 rolling window 기반 
    Sharpe, Volatility, Sortino, Calmar 계산

    Args:
        returns (pd.Series): 일간 수익률
        windows (list): rolling window list (e.g., [20, 252])
        risk_free_rate (float): 무위험 수익률 (연환산)
        annual_factor (int): 연환산 계수 (default 252)

    Returns:
        pd.DataFrame: 원본 daily_return + 모든 rolling metric columns
    """
    result = pd.DataFrame({'daily_return': returns})
    result["tic"] = name

    for window in windows:
        result[f'sharpe_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_sharpe_ratio(x, risk_free_rate, annual_factor),
            raw=False
        )
        result[f'vol_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_volatility(x, annual_factor),
            raw=False
        )
        result[f'sortino_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_annualized_sortino_ratio(x, risk_free_rate, annual_factor),
            raw=False
        )
        result[f'calmar_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_calmar_ratio(x, annual_factor),
            raw=False
        )
    if shift_features:
        feature_cols = [col for col in result.columns if col not in ['daily_return', 'tic']]
        result[feature_cols] = result[feature_cols].shift(1)
        
    return result

In [115]:
rp_eval = calculate_rolling_metrics(rp_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_rp")
minvar_eval = calculate_rolling_metrics(minvar_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_minvar")
ms_eval = calculate_rolling_metrics(ms_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_ms")
paa_eval = calculate_rolling_metrics(paa_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_paa")
equal_eval = calculate_rolling_metrics(equal_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="equal")

In [116]:
risk_df = pd.DataFrame({
    'close' : risk_cum,
    "tic" : "risk_parity"})

minvar_df = pd.DataFrame({
    'close' : minvar_cum,
    "tic" : "minvar"})

ms_df = pd.DataFrame({
    'close' : ms_cum,
    "tic" : "max_sharpe"})

paa_df = pd.DataFrame({
    'close' : paa_cum,
    "tic" : "paa"})

In [117]:
import pandas as pd

def calculate_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    주어진 데이터프레임에 대해 모멘텀 관련 특성 생성.
    
    Args:
        df (pd.DataFrame): 종가(close) 컬럼을 포함한 데이터프레임
        close_col (str): 종가 컬럼 이름 (default: 'close')
        
    Returns:
        pd.DataFrame: 모멘텀 특성이 추가된 데이터프레임
    """
    data = df.copy()
    
    # 설정할 period 리스트
    periods = [20, 40, 60, 80, 100, 120, 140, 160, 180, 220, 240, 252]
    columns_list = ["return_1m", "return_3m", "return_6m", "return_12m", "return_avg", "mom_12m", "mom_score"]

    # 기간별 이전 종가 저장
    for idx, period in enumerate(periods):
        col_name = f'close_{idx+1}_month'
        data[col_name] = data['close'].shift(period)
        columns_list.append(col_name)
    
    # 모멘텀 및 수익률 계산
    data['return_1m'] = (data['close'] - data['close_1_month']) / data['close_1_month']
    data['return_3m'] = (data['close'] - data['close_3_month']) / data['close_3_month']
    data['return_6m'] = (data['close'] - data['close_6_month']) / data['close_6_month']
    data['return_12m'] = (data['close'] - data['close_12_month']) / data['close_12_month']
    data['return_avg'] = data[['return_1m', 'return_3m', 'return_6m', 'return_12m']].mean(axis=1)
    data["mom_score"] = (
        12 * data["return_1m"] +
        4 * data["return_3m"] +
        2 * data["return_6m"] +
        1 * data["return_12m"]
    ) / 19

    # 평균 return
    data['return_avg'] = data[['return_1m', 'return_3m', 'return_6m', 'return_12m']].mean(axis=1)
    
    # 모멘텀 점수 (mom_score)
    data['mom_score'] = (
        12 * data['return_1m'] +
        4 * data['return_3m'] +
        2 * data['return_6m'] +
        1 * data['return_12m']
    ) / 19
    
    return data


In [118]:
risk_df = calculate_momentum_features(risk_df)
minvar_df = calculate_momentum_features(minvar_df)
ms_df = calculate_momentum_features(ms_df)
paa_df = calculate_momentum_features(paa_df)

In [119]:
risk_df["ret"] = risk_df["close"].pct_change().shift(1)
minvar_df["ret"] = minvar_df["close"].pct_change().shift(1)
ms_df["ret"] = ms_df["close"].pct_change().shift(1)
paa_df["ret"] = paa_df["close"].pct_change().shift(1)


In [120]:
eval_feature = ['vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [121]:
risk_df = pd.concat([risk_df, rp_eval[eval_feature]], axis=1)
minvar_df = pd.concat([minvar_df, minvar_eval[eval_feature]], axis=1)
ms_df = pd.concat([ms_df, ms_eval[eval_feature]], axis=1)
paa_df = pd.concat([paa_df, paa_eval[eval_feature]], axis=1)

In [122]:
# portfolio_df = pd.concat([future_risk_df, index_risk_df, future_ms_df, index_ms_df, future_paa_df, index_paa_df], axis=0)


In [123]:
portfolio_df = pd.concat([risk_df, minvar_df, ms_df, paa_df], axis=0)

In [124]:
portfolio_df.reset_index(inplace=True)

In [125]:
portfolio_df.rename(columns={"index": "date"}, inplace=True)

In [126]:
portfolio_df.tic.unique()

array(['risk_parity', 'minvar', 'max_sharpe', 'paa'], dtype=object)

In [127]:
portfolio_df.columns

Index(['date', 'close', 'tic', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score', 'ret', 'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252'],
      dtype='object')

In [128]:
select_features = ['return_1m',
       'return_3m', 'return_6m', 'return_12m', 'return_avg',
       'ret', 'vol_20', 'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [129]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp[select_features].describe())

risk_parity
         return_1m    return_3m    return_6m   return_12m   return_avg  \
count  5711.000000  5671.000000  5611.000000  5479.000000  5711.000000   
mean      0.005899     0.018057     0.037553     0.081537     0.034739   
std       0.043067     0.079119     0.120648     0.181760     0.086054   
min      -0.321162    -0.409053    -0.473276    -0.487938    -0.404640   
25%      -0.016338    -0.020766    -0.018620    -0.048607    -0.010229   
50%       0.009447     0.021836     0.042592     0.090701     0.042823   
75%       0.031247     0.065176     0.095901     0.187575     0.085402   
max       0.184196     0.427170     0.507004     0.683816     0.271563   

               ret       vol_20   sharpe_252      vol_252  sortino_252  \
count  5729.000000  5711.000000  5479.000000  5479.000000  5479.000000   
mean      0.000283     0.118509     0.873648     0.128331     1.270414   
std       0.008673     0.068304     1.441272     0.052385     2.131152   
min      -0.082540     0.

In [130]:
portfolio_df.sort_values(by=["date", "tic"], inplace=True)

In [131]:
portfolio_df.reset_index(drop=True, inplace=True)

In [132]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp.describe())

max_sharpe


             close  close_1_month  close_2_month  close_3_month  \
count  5731.000000    5711.000000    5691.000000    5671.000000   
mean      1.802145       1.800845       1.799749       1.798653   
std       0.519045       0.519486       0.520067       0.520654   
min       0.980140       0.980140       0.980140       0.980140   
25%       1.315708       1.314597       1.313375       1.312376   
50%       1.750075       1.749209       1.746387       1.744384   
75%       2.239500       2.245856       2.250901       2.254672   
max       3.130891       3.130891       3.130891       3.130891   

       close_4_month  close_5_month  close_6_month  close_7_month  \
count    5651.000000    5631.000000    5611.000000    5591.000000   
mean        1.797489       1.796540       1.795924       1.795261   
std         0.521204       0.521881       0.522705       0.523518   
min         0.980140       0.980140       0.980140       0.980140   
25%         1.311822       1.311431       1.310591 

In [133]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp.isna().sum())

max_sharpe
date                0
close               0
tic                 0
close_1_month      20
close_2_month      40
close_3_month      60
close_4_month      80
close_5_month     100
close_6_month     120
close_7_month     140
close_8_month     160
close_9_month     180
close_10_month    220
close_11_month    240
close_12_month    252
return_1m          20
return_3m          60
return_6m         120
return_12m        252
return_avg         20
mom_score         252
ret                 2
vol_20             20
sharpe_252        252
vol_252           252
sortino_252       252
calmar_252        252
dtype: int64
minvar
date                0
close               0
tic                 0
close_1_month      20
close_2_month      40
close_3_month      60
close_4_month      80
close_5_month     100
close_6_month     120
close_7_month     140
close_8_month     160
close_9_month     180
close_10_month    220
close_11_month    240
close_12_month    252
return_1m          20
return_3m          60
r

In [134]:
portfolio_na = portfolio_df.dropna()

In [135]:
portfolio_na.reset_index(drop=True, inplace=True)

In [136]:
for tic  in portfolio_na.tic.unique():
    imp = portfolio_na[portfolio_na.tic == tic].copy()
    print(tic)
    print(imp.date.min())
    print(imp.date.max())

max_sharpe
2003-02-19
2024-12-31
minvar
2003-02-19
2024-12-31
paa
2003-02-19
2024-12-31
risk_parity
2003-02-19
2024-12-31


In [137]:
portfolio_na.rename(columns={"tic": "ticker"}, inplace=True)

/tmp/ipykernel_3608481/1574858568.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  portfolio_na.rename(columns={"tic": "ticker"}, inplace=True)


In [138]:
TRAIN_START_DATE = '2003-01-01'
TRAIN_END_DATE = '2013-12-31'
VAILD_START_DATE = '2014-01-01'
VAILD_END_DATE = '2018-12-31'
TEST_START_DATE = '2019-01-01'
TEST_END_DATE = '2024-12-31'

In [139]:
def data_split(df, start, end, target_date_col="date"):
    """
    split the dataset into training or testing using date
    :param data: (df) pandas dataframe, start, end
    :return: (df) pandas dataframe
    """
    data = df[(df[target_date_col] >= start) & (df[target_date_col] <= end)]
    data = data.sort_values([target_date_col, "ticker"], ignore_index=True)
    # data.index = data[target_date_col].factorize()[0]
    return data

In [140]:
train = data_split(portfolio_na, TRAIN_START_DATE,TRAIN_END_DATE)
vaild = data_split(portfolio_na, VAILD_START_DATE,VAILD_END_DATE)
test = data_split(portfolio_na, TEST_START_DATE,TEST_END_DATE)

In [141]:
# def min_max_normalize_by_ticker_train_test(train_df, vaild_df, test_df, columns):
#     """
#     주어진 컬럼들을 train 기준으로 티커별 min-max 정규화
    
#     Parameters:
#         train_df (DataFrame): 학습용 데이터
#         test_df (DataFrame): 테스트용 데이터
#         columns (list): 정규화할 컬럼 리스트
        
#     Returns:
#         train_df, test_df: 정규화된 결과가 포함된 데이터프레임
#     """
#     # 티커별로 정규화 통계 계산
#     stats = train_df.groupby("ticker")[columns].agg(["min", "max"])
    
#     # 컬럼명 정리 (MultiIndex → flat column name)
#     stats.columns = [f"{col}_{stat}" for col, stat in stats.columns]

#     # train/test에 붙이기
#     train_df = train_df.merge(stats, on="ticker", how="left")
#     vaild_df = vaild_df.merge(stats, on="ticker", how="left")
#     test_df = test_df.merge(stats, on="ticker", how="left")

#     # 컬럼별 정규화 수행
#     for col in columns:
#         min_col = f"{col}_min"
#         max_col = f"{col}_max"
#         norm_col = f"{col}_norm"

#         train_df[norm_col] = (train_df[col] - train_df[min_col]) / (train_df[max_col] - train_df[min_col])
#         vaild_df[norm_col] = (vaild_df[col] - vaild_df[min_col]) / (vaild_df[max_col] - vaild_df[min_col])
#         test_df[norm_col] = (test_df[col] - test_df[min_col]) / (test_df[max_col] - test_df[min_col])

#     # 불필요한 min/max 컬럼 제거
#     cols_to_drop = [f"{col}_min" for col in columns] + [f"{col}_max" for col in columns]
#     train_df.drop(columns=cols_to_drop, inplace=True)
#     vaild_df.drop(columns=cols_to_drop, inplace=True)
#     test_df.drop(columns=cols_to_drop, inplace=True)

#     return train_df, vaild_df, test_df


In [142]:
def min_max_normalize_global_train_test(train_df, valid_df, test_df, columns):
    """
    전체 train 데이터 기준으로 주어진 컬럼들에 대해 min-max 정규화
    
    Parameters:
        train_df (DataFrame): 학습용 데이터
        valid_df (DataFrame): 검증용 데이터
        test_df (DataFrame): 테스트용 데이터
        columns (list): 정규화할 컬럼 리스트
        
    Returns:
        train_df, valid_df, test_df: 정규화된 결과가 포함된 데이터프레임
    """
    # train 데이터 전체 기준으로 min/max 계산
    stats = train_df[columns].agg(["min", "max"])
    
    for col in columns:
        min_val = stats.loc["min", col]
        max_val = stats.loc["max", col]
        norm_col = f"{col}_norm"
        
        train_df[norm_col] = (train_df[col] - min_val) / (max_val - min_val)
        valid_df[norm_col] = (valid_df[col] - min_val) / (max_val - min_val)
        test_df[norm_col] = (test_df[col] - min_val) / (max_val - min_val)
    
    return train_df, valid_df, test_df


In [143]:
columns_to_normalize = ["ret", 'close', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score',  'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [57]:
# train_df, vaild_df, test_df = min_max_normalize_by_ticker_train_test(train, vaild, test, columns_to_normalize)

In [144]:
train_df, vaild_df, test_df = min_max_normalize_global_train_test(train, vaild, test, columns_to_normalize)

In [145]:
train_df[['ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm']].describe()

,ret_norm,close_norm,return_1m_norm,return_3m_norm,return_6m_norm,return_12m_norm,return_avg_norm,mom_score_norm,vol_20_norm,sharpe_252_norm,vol_252_norm,sortino_252_norm
count,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000,10872.000000
mean,0.500879,0.368422,0.537376,0.481466,0.476134,0.507985,0.551218,0.581322,0.224698,0.413357,0.321069,0.363120
std,0.062473,0.201607,0.085573,0.108571,0.119920,0.153075,0.123860,0.099116,0.129240,0.189583,0.217429,0.178900
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.473055,0.192090,0.494903,0.427050,0.421348,0.429270,0.488905,0.534055,0.132762,0.273449,0.153973,0.231908
50%,0.503625,0.390211,0.543753,0.483416,0.473752,0.522179,0.557823,0.585431,0.189331,0.411258,0.269997,0.354328
75%,0.532286,0.513417,0.585922,0.541398,0.542365,0.604102,0.631833,0.639796,0.279838,0.553851,0.466530,0.486833
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [146]:
for tic in test_df.ticker.unique():
    imp = train_df[train_df.ticker == tic].copy()
    print(tic)
    print(imp.describe())


max_sharpe
             close  close_1_month  close_2_month  close_3_month  \
count  2718.000000    2718.000000    2718.000000    2718.000000   
mean      1.966360       1.955953       1.945259       1.934425   
std       0.526150       0.528247       0.530788       0.533768   
min       1.013740       1.013740       1.013740       1.013740   
25%       1.483544       1.474339       1.466626       1.460190   
50%       1.991601       1.985068       1.972635       1.967868   
75%       2.423118       2.414747       2.408939       2.403097   
max       3.130891       3.130891       3.130891       3.130891   

       close_4_month  close_5_month  close_6_month  close_7_month  \
count    2718.000000    2718.000000    2718.000000    2718.000000   
mean        1.923839       1.913943       1.903629       1.893545   
std         0.536731       0.539361       0.541481       0.544166   
min         1.013740       1.013740       1.013740       1.013740   
25%         1.452585       1.444001     

In [147]:
train_df.to_csv("../data/portfolio_price/train_total_asset_portfolio_monthly_v1.csv", index=False)
vaild_df.to_csv("../data/portfolio_price/vaild_total_asset_portfolio_monthly_v1.csv", index=False)
test_df.to_csv("../data/portfolio_price/test_total_asset_portfolio_monthly_v1.csv", index=False)

In [148]:
train_df.columns

Index(['date', 'close', 'ticker', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score', 'ret', 'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252', 'ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm',
       'calmar_252_norm'],
      dtype='object')

In [149]:
select_features = ['ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm',
       'calmar_252_norm']

In [150]:
tot_data = pd.concat([train_df, vaild_df, test_df])

In [151]:
tot_data = tot_data.sort_values(["date", "ticker"])

In [152]:
describe_df = tot_data[select_features].describe(include='all')

In [153]:
describe_df = vaild_df[select_features].describe(include='all')

In [154]:
# 고유 날짜/종목 추출 및 정렬
dates = np.sort(tot_data['date'].unique())
tics = np.sort(tot_data['ticker'].unique())

In [155]:
# 인덱스 매핑 (빠른 접근용)
date2idx = {d: i for i, d in enumerate(dates)}
tic2idx = {t: i for i, t in enumerate(tics)}

In [156]:
state_array = np.zeros((len(dates), len(tics), len(select_features)), dtype=np.float32)

In [157]:
for row in tot_data.itertuples():
    d_idx = date2idx[row.date]
    t_idx = tic2idx[row.ticker]
    f_vals = [getattr(row, f) for f in select_features]
    state_array[d_idx, t_idx, :] = np.nan_to_num(f_vals)  # NaN은 0으로

In [158]:
state_array[0]

array([[0.52454066, 0.05266879, 0.56561184, 0.57070166, 0.47606272,
        0.53918004, 0.5933397 , 0.62728447, 0.17319444, 0.4590155 ,
        0.15763204, 0.39856285, 0.19112083],
       [0.5052916 , 0.04988786, 0.5095343 , 0.5160665 , 0.47956786,
        0.51940084, 0.5614266 , 0.5768683 , 0.11008433, 0.5351271 ,
        0.05420408, 0.4710987 , 0.21094742],
       [0.6100688 , 0.08173236, 0.7072097 , 0.7105302 , 0.5550068 ,
        0.6090118 , 0.71574944, 0.7750132 , 0.3540435 , 0.4763887 ,
        0.31493944, 0.4333538 , 0.21528666],
       [0.5231913 , 0.01237122, 0.49219704, 0.4831367 , 0.43249178,
        0.4311548 , 0.49578023, 0.53805256, 0.12383573, 0.26497394,
        0.18323201, 0.22483054, 0.06648669]], dtype=float32)

In [159]:
tot_data[tot_data["date"]=="2018-12-31"][select_features]

,ret_norm,close_norm,return_1m_norm,return_3m_norm,return_6m_norm,return_12m_norm,return_avg_norm,mom_score_norm,vol_20_norm,sharpe_252_norm,vol_252_norm,sortino_252_norm,calmar_252_norm
5000,0.299215,0.072382,0.307433,0.126032,0.187754,0.160631,0.162264,0.246562,0.531914,-0.039297,0.318634,0.000691,0.009884
5001,0.512827,0.787025,0.530154,0.418733,0.413576,0.381687,0.458082,0.532590,0.148906,0.174701,0.063629,0.148806,0.041576
5002,-0.048415,0.169143,0.149386,0.099472,0.151944,0.156371,0.110539,0.137832,1.258701,0.081097,0.715715,0.095178,0.011490
5003,0.559344,0.498949,0.471368,0.338436,0.348587,0.323406,0.377157,0.455129,0.245459,0.079878,0.147682,0.071094,0.024079


In [160]:
TRAIN_START_DATE = '2003-01-01'
TRAIN_END_DATE = '2013-12-31'
VAILD_START_DATE = '2014-01-01'
VAILD_END_DATE = '2018-12-31'
TEST_START_DATE = '2019-01-01'
TEST_END_DATE = '2024-12-31'

In [161]:
valid_start_date = "2011-01-03"
test_start_date = "2017-01-03"

In [162]:
date2idx['2014-01-02']

2718

In [163]:
train = state_array[:2718]

In [164]:
valid = state_array[2718:3969]

In [165]:
test = state_array[3969:]

In [166]:
import torch

In [167]:
train_tensor = torch.from_numpy(train)
valid_tensor = torch.from_numpy(valid)
test_tensor = torch.from_numpy(test)

In [168]:
train.shape

(2718, 4, 13)

In [169]:
train_tensor.shape

torch.Size([2718, 4, 13])

In [170]:
valid_tensor.shape

torch.Size([1251, 4, 13])

In [171]:
torch.save(train_tensor, "../data/portfolio_price/concat_portfolio_train_monthly_v1.pt")
torch.save(test_tensor,  "../data/portfolio_price/concat_portfolio_test_monthly_v1.pt")
torch.save(valid_tensor, "../data/portfolio_price/concat_portfolio_valid_monthly_v1.pt")

In [172]:
def calculate_annual_return(returns, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    cumulative = np.prod(1 + returns)
    n_periods = len(returns)
    return cumulative ** (annual_factor / n_periods) - 1


def calculate_max_drawdown(returns):
    returns = np.array(returns, dtype=np.float64)
    cumulative_returns = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdowns = (cumulative_returns - running_max) / running_max
    return abs(np.min(drawdowns))

def calculate_volatility(returns, annual_factor=252):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1) * np.sqrt(annual_factor)

def calculate_sharpe_ratio(returns, risk_free_rate=0.02, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    annual_return = calculate_annual_return(returns, annual_factor)
    annual_std_dev = calculate_volatility(returns, annual_factor)
    excess_return =  (annual_return - risk_free_rate) 
    # 연환산 수익률과 표준편차
    return excess_return / annual_std_dev if annual_std_dev != 0 else 0.0


def calculate_cumulative_return(returns):
    returns = np.array(returns, dtype=np.float64)
    return np.prod(1 + returns) - 1

def calculate_volatility(returns, annual_factor=252):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1) * np.sqrt(annual_factor)


# 성과 지표 계산 함수 (파라미터화)
def calculate_performance_metrics(returns, annual_factor=252, risk_free_rate=0.0):
    cumulative_returns = (1 + returns).cumprod()
    # cagr = calculate_cagr(returns, annual_factor)
    annual_return = calculate_annual_return(returns, annual_factor)
    sharpe_ratio = calculate_sharpe_ratio(returns, risk_free_rate, annual_factor)
    mdd = calculate_max_drawdown(returns)
    volatility = calculate_volatility(returns, annual_factor)

    return {
        'Cumulative Return': cumulative_returns.iloc[-1] - 1,
        'Annual Return': annual_return,
        # 'CAGR': cagr,
        'Volatility': volatility,
        'Sharpe Ratio': sharpe_ratio,
        'MDD': mdd
    }


# 성과 지표 요약 생성 함수 (파라미터 전달)
def get_performance_summary(returns_dict, annual_factor=252, risk_free_rate=0.0):
    metrics_df = pd.DataFrame()
    for name, returns in returns_dict.items():
        metrics_df[name] = calculate_performance_metrics(returns, annual_factor, risk_free_rate)
    return metrics_df.T


In [173]:
metrics_summary = get_performance_summary(
    strategy_returns,
    annual_factor=annual_factor,
    risk_free_rate=0.0
)

display(metrics_summary)


,Cumulative Return,Annual Return,Volatility,Sharpe Ratio,MDD
risk_parity,3.050437,0.063439,0.137658,0.460848,0.517716
minvar,5.754640,0.087624,0.098399,0.890500,0.346831
max_shapre,1.161621,0.034477,0.170528,0.202176,0.686945
paa,1.603418,0.042971,0.202794,0.211893,0.757277
Equal Weight,3.818659,0.071591,0.148739,0.481322,0.521208


In [88]:
strategy_returns

{'risk_parity': 2002-02-05   -0.004431
 2002-02-06   -0.003447
 2002-02-07    0.002515
 2002-02-08    0.006796
 2002-02-11    0.003493
                 ...   
 2024-12-24    0.004320
 2024-12-26    0.002367
 2024-12-27   -0.002598
 2024-12-30   -0.002941
 2024-12-31    0.000488
 Length: 5731, dtype: float64,
 'minvar': 2002-02-05    0.007067
 2002-02-06   -0.000896
 2002-02-07   -0.000863
 2002-02-08    0.007633
 2002-02-11   -0.005547
                 ...   
 2024-12-24    0.004868
 2024-12-26    0.002580
 2024-12-27   -0.002314
 2024-12-30    0.003596
 2024-12-31    0.003003
 Length: 5731, dtype: float64,
 'max_shapre': 2002-02-05   -0.007578
 2002-02-06    0.004537
 2002-02-07    0.012914
 2002-02-08    0.007345
 2002-02-11    0.007395
                 ...   
 2024-12-24    0.010024
 2024-12-26   -0.007023
 2024-12-27   -0.013622
 2024-12-30    0.012663
 2024-12-31   -0.001817
 Length: 5731, dtype: float64,
 'paa': 2002-02-05   -0.000850
 2002-02-06   -0.002986
 2002-02-07    0.0033